# 22 — Explainability and Model Interpretation for Vision Models

In the previous notebook, we learned how to evaluate classifiers using confusion matrices, threshold-based metrics, ROC/PR curves, calibration, and error analysis.

Now we will study another important question:

> **What parts of an image are influencing a vision model's prediction?**

This is the domain of **model interpretation** and **explainability**.

For vision models, common interpretation tools include:

- Feature-map visualization
- Activation maps
- Input-gradient saliency
- Grad-CAM
- Occlusion sensitivity
- Failure-case inspection

These methods can be extremely useful for debugging.

But they must be interpreted carefully.

A heatmap is **not** proof that:

- The model reasons like a human
- The highlighted region is causal
- The model is clinically correct
- The explanation is stable
- The model will generalize

## In this notebook, we will study:

1. Why model interpretation matters
2. Feature-map visualization
3. CNN activation maps
4. Saliency maps
5. Input gradients
6. Grad-CAM intuition
7. Implementing Grad-CAM
8. Interpreting heatmaps carefully
9. Occlusion sensitivity
10. Confidence vs explanation
11. Failure-case inspection
12. Shortcut learning
13. Spurious correlations
14. Explainability pitfalls
15. Ultrasound-specific interpretation concerns
16. Using explanations as debugging tools, not proof of causality
17. Common implementation mistakes
18. Practice exercises

## Main Goal

By the end of this notebook, you should understand this interpretation pipeline:

$$
\boxed{
\text{Image}
\rightarrow
\text{Model}
\rightarrow
\text{Prediction}
\rightarrow
\text{Explanation Method}
\rightarrow
\text{Inspect What Influenced the Prediction}
}
$$

The key principle is:

> **Interpretability methods are diagnostic tools. They provide evidence about model behavior, not guaranteed explanations of causal reasoning.**


In [ ]:
import copy
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. Why Model Interpretation Matters

A classifier can achieve strong accuracy while learning the wrong signal.

For example, a medical-image model may accidentally use:

- Scanner text
- Measurement markers
- Borders
- Cropping patterns
- Site-specific formatting
- Probe annotations
- Acquisition artifacts

instead of the clinically relevant anatomy.

Interpretation methods can help us ask:

> **Where is the model looking?**

and:

> **Does the model appear to rely on suspicious image regions?**


# 2. Interpretation Is Not the Same as Evaluation

Evaluation asks:

> How well does the model perform?

Interpretation asks:

> What image information appears to influence its prediction?

Both are necessary.

A model can be:

- Accurate but using shortcuts
- Poorly calibrated but visually focused
- Correct for the wrong reason
- Wrong while attending to a plausible region

Interpretation should complement, not replace, quantitative evaluation.


# 3. A Small Vision Model for This Notebook

We will use a small CNN so that every internal activation can be inspected directly.

Input:

$$
(N,\ 1,\ 64,\ 64)
$$

Architecture:

$$
1
\rightarrow
8
\rightarrow
16
\rightarrow
32
\rightarrow
3
$$

The final output contains three class logits.


In [ ]:
class ExplainableCNN(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            1,
            8,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            8,
            16,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):
        x = F.relu(
            self.conv1(x)
        )

        x = self.pool(x)

        x = F.relu(
            self.conv2(x)
        )

        x = self.pool(x)

        x = F.relu(
            self.conv3(x)
        )

        x = F.adaptive_avg_pool2d(
            x,
            output_size=1
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(
            x
        )

model = ExplainableCNN(
    num_classes=3
)

print(model)


# 4. Why Use Adaptive Average Pooling?

The final convolution output has shape:

$$
(N,\ 32,\ H,\ W)
$$

Adaptive average pooling converts it to:

$$
(N,\ 32,\ 1,\ 1)
$$

Then flattening gives:

$$
(N,\ 32)
$$

This makes the classifier less dependent on one exact input resolution.


# 5. Synthetic Vision Data

We will create simple grayscale images containing one of three structures:

- Class 0 — vertical bright bar
- Class 1 — horizontal bright bar
- Class 2 — bright square

The goal is not realism.

The goal is to create a visual problem where we know what signal the model *should* learn.


In [ ]:
def make_pattern_image(
    class_index,
    image_size=64,
    noise_std=0.12
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    shift_x = torch.randint(
        -8,
        9,
        (1,)
    ).item()

    shift_y = torch.randint(
        -8,
        9,
        (1,)
    ).item()

    x_center = center + shift_x
    y_center = center + shift_y

    if class_index == 0:
        start = max(
            0,
            x_center - 3
        )

        end = min(
            image_size,
            x_center + 3
        )

        image[
            :,
            8:image_size - 8,
            start:end
        ] = 1.0

    elif class_index == 1:
        start = max(
            0,
            y_center - 3
        )

        end = min(
            image_size,
            y_center + 3
        )

        image[
            :,
            start:end,
            8:image_size - 8
        ] = 1.0

    elif class_index == 2:
        half = 8

        y0 = max(
            0,
            y_center - half
        )

        y1 = min(
            image_size,
            y_center + half
        )

        x0 = max(
            0,
            x_center - half
        )

        x1 = min(
            image_size,
            x_center + half
        )

        image[
            :,
            y0:y1,
            x0:x1
        ] = 1.0

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    image += (
        torch.randn_like(
            image
        )
        * noise_std
    )

    return image.clamp(
        0.0,
        1.0
    )


# 6. Visualizing the Three Synthetic Classes


In [ ]:
torch.manual_seed(42)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

for class_index in range(3):
    image = make_pattern_image(
        class_index
    )

    axes[
        class_index
    ].imshow(
        image.squeeze(0).numpy(),
        cmap="gray"
    )

    axes[
        class_index
    ].set_title(
        f"Class {class_index}"
    )

    axes[
        class_index
    ].axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 7. Creating a Small Training Dataset


In [ ]:
torch.manual_seed(42)

train_images = []
train_targets = []

for class_index in range(3):
    for _ in range(180):
        train_images.append(
            make_pattern_image(
                class_index
            )
        )

        train_targets.append(
            class_index
        )

train_images = torch.stack(
    train_images
)

train_targets = torch.tensor(
    train_targets,
    dtype=torch.long
)

permutation = torch.randperm(
    len(train_images)
)

train_images = train_images[
    permutation
]

train_targets = train_targets[
    permutation
]

print(
    "Images:",
    train_images.shape
)

print(
    "Targets:",
    train_targets.shape
)


# 8. Training the Demonstration CNN

We will briefly train the model so its interpretation maps are meaningful.

This is only a teaching example.


In [ ]:
torch.manual_seed(42)

model = ExplainableCNN(
    num_classes=3
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-3
)

batch_size = 64

for epoch in range(12):
    model.train()

    permutation = torch.randperm(
        len(train_images)
    )

    epoch_loss = 0.0
    correct = 0

    for start in range(
        0,
        len(train_images),
        batch_size
    ):
        indices = permutation[
            start:start + batch_size
        ]

        images = train_images[
            indices
        ]

        targets = train_targets[
            indices
        ]

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        epoch_loss += (
            loss.item()
            * len(indices)
        )

        correct += (
            logits.argmax(
                dim=1
            )
            == targets
        ).sum().item()

    if (
        epoch == 0
        or
        (epoch + 1) % 4 == 0
    ):
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Loss {epoch_loss / len(train_images):.4f} | "
            f"Accuracy {correct / len(train_images):.3f}"
        )


# 9. Selecting an Example Image


In [ ]:
example_image = make_pattern_image(
    class_index=2
).unsqueeze(
    0
)

model.eval()

with torch.inference_mode():
    example_logits = model(
        example_image
    )

    example_probs = torch.softmax(
        example_logits,
        dim=1
    )

    example_prediction = (
        example_logits.argmax(
            dim=1
        ).item()
    )

print(
    "Input:",
    example_image.shape
)

print(
    "Prediction:",
    example_prediction
)

print(
    "Probabilities:",
    example_probs
)


# 10. Feature Maps

A convolutional layer produces multiple channels.

For example:

$$
(N,\ 8,\ 64,\ 64)
$$

contains:

$$
8
$$

feature maps per image.

Each channel is the response of a learned filter across spatial positions.


# 11. Extracting the First-Layer Activations

We can manually run the image through the first convolution and ReLU.


In [ ]:
model.eval()

with torch.inference_mode():
    first_activation = F.relu(
        model.conv1(
            example_image
        )
    )

print(
    "Activation shape:",
    first_activation.shape
)


# 12. Visualizing Feature Maps

We can display each channel as an image.

Important:

> A feature map is not automatically interpretable as a specific human concept.

Some channels may resemble:

- Edges
- Local contrast
- Texture
- Shape fragments

But we should avoid assigning semantic meaning without evidence.


In [ ]:
num_maps = min(
    8,
    first_activation.shape[1]
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(10, 5)
)

for channel, axis in enumerate(
    axes.flat
):
    if channel < num_maps:
        axis.imshow(
            first_activation[
                0,
                channel
            ].numpy(),
            cmap="viridis"
        )

        axis.set_title(
            f"Channel {channel}"
        )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 13. Deeper Activations

As we go deeper:

- Spatial resolution often decreases
- Channel count increases
- Features become more task-specific

Let's inspect the output of the third convolution before global pooling.


In [ ]:
model.eval()

with torch.inference_mode():
    x = F.relu(
        model.conv1(
            example_image
        )
    )

    x = model.pool(
        x
    )

    x = F.relu(
        model.conv2(
            x
        )
    )

    x = model.pool(
        x
    )

    third_activation = F.relu(
        model.conv3(
            x
        )
    )

print(
    "Third activation:",
    third_activation.shape
)


# 14. Visualizing Deeper Feature Maps


In [ ]:
fig, axes = plt.subplots(
    2,
    4,
    figsize=(10, 5)
)

for channel, axis in enumerate(
    axes.flat
):
    axis.imshow(
        third_activation[
            0,
            channel
        ].numpy(),
        cmap="viridis"
    )

    axis.set_title(
        f"Channel {channel}"
    )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 15. Feature Maps Are Internal Representations

Feature-map visualization answers:

> What spatial pattern activated this channel?

It does **not** directly answer:

> Which region was important for the final prediction?

For that, we need methods that connect the prediction score back to the input or intermediate features.


# 16. Saliency Maps

A simple saliency map uses the gradient of a class score with respect to the input image.

For class score:

$$
s_c
$$

and input pixels:

$$
x
$$

we compute:

$$
\boxed{
\frac{
\partial s_c
}{
\partial x
}
}
$$

Large gradient magnitude means:

> A small change in that input pixel could strongly change the selected class score locally.


# 17. Input Gradients

To compute an input gradient:

1. Clone the input
2. Set `requires_grad=True`
3. Run forward pass
4. Select a target class score
5. Call `backward()`
6. Read `input.grad`


In [ ]:
saliency_input = (
    example_image
    .detach()
    .clone()
)

saliency_input.requires_grad_(
    True
)

model.zero_grad(
    set_to_none=True
)

logits = model(
    saliency_input
)

target_class = logits.argmax(
    dim=1
).item()

score = logits[
    0,
    target_class
]

score.backward()

input_gradient = (
    saliency_input.grad
    .detach()
)

print(
    "Gradient shape:",
    input_gradient.shape
)


# 18. Converting Input Gradient to a Saliency Map

For a one-channel image, a simple saliency map is:

$$
\left|
\frac{
\partial s_c
}{
\partial x
}
\right|
$$


In [ ]:
saliency = input_gradient.abs()

saliency = (
    saliency
    / (
        saliency.max()
        + 1e-8
    )
)

print(
    "Saliency:",
    saliency.shape
)


# 19. Visualizing the Saliency Map


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(8, 4)
)

axes[0].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Input"
)

axes[0].axis(
    "off"
)

axes[1].imshow(
    saliency[
        0,
        0
    ].numpy(),
    cmap="hot"
)

axes[1].set_title(
    f"Input-gradient saliency\nClass {target_class}"
)

axes[1].axis(
    "off"
)

plt.tight_layout()
plt.show()


# 20. What Does Saliency Mean?

A gradient saliency map is a **local sensitivity map**.

It tells us:

> Which pixels would most strongly change the selected score under a very small perturbation?

It does not necessarily tell us:

- Which pixels caused the prediction
- Which pixels the model "understood"
- Which region is clinically meaningful


# 21. Saliency Can Be Noisy

Raw input gradients can look noisy because:

- Deep networks are highly nonlinear
- Small local derivatives can fluctuate
- ReLU boundaries can make gradients unstable
- Input preprocessing affects gradient scale

This is one reason several explanation methods exist.


# 22. Saliency for RGB Images

For RGB input:

$$
(3,\ H,\ W)
$$

you obtain a gradient per channel.

A common visualization reduces channels using:

$$
max_c
\left|
\frac{
\partial s
}{
\partial x_c
}
\right|
$$

or another aggregation.

The channel-reduction choice is itself part of the explanation design.


# 23. Grad-CAM Intuition

Grad-CAM stands for:

> **Gradient-weighted Class Activation Mapping**

Instead of taking gradients directly with respect to pixels, Grad-CAM uses:

- Deep convolutional feature maps
- Gradients of the selected class score with respect to those feature maps

It produces a coarse spatial heatmap showing regions associated with the selected class score.


# 24. Grad-CAM Mathematics

Suppose the target convolutional layer produces feature maps:

$$
A^k
$$

where:

$$
k
$$

indexes channels.

For class:

$$
c
$$

we compute gradients:

$$
\frac{
\partial s_c
}{
\partial A^k_{ij}
}
$$

Then average those gradients spatially:

$$
\boxed{
\alpha_k^c
=
\frac{1}{Z}
\sum_i
\sum_j
\frac{
\partial s_c
}{
\partial A^k_{ij}
}
}
$$

These values tell us how strongly each feature map contributes to the selected class.


# 25. Grad-CAM Heatmap

The Grad-CAM map is constructed as:

$$
\boxed{
L_{GradCAM}^{c}
=
ReLU
\left(
\sum_k
\alpha_k^c
A^k
\right)
}
$$

The ReLU keeps positive evidence for the selected class.

The result is then resized to the input image resolution.


# 26. Why Use a Late Convolutional Layer?

Early layers have:

- High spatial detail
- Low semantic abstraction

Late layers have:

- Lower spatial resolution
- More task-specific features

Grad-CAM is commonly applied to one of the last convolutional layers because it balances:

- Spatial localization
- Class-specific information


# 27. A Reusable Grad-CAM Helper

We will implement Grad-CAM without external explainability libraries.

The helper will:

1. Attach a forward hook to the target layer
2. Save its activation tensor
3. Retain the activation gradient
4. Backpropagate the selected class score
5. Compute channel weights
6. Build the heatmap


In [ ]:
class GradCAM:
    def __init__(
        self,
        model,
        target_layer
    ):
        self.model = model
        self.target_layer = target_layer
        self.activations = None

        self.handle = (
            target_layer
            .register_forward_hook(
                self._save_activation
            )
        )

    def _save_activation(
        self,
        module,
        inputs,
        output
    ):
        self.activations = output

        if output.requires_grad:
            output.retain_grad()

    def generate(
        self,
        image,
        class_index=None
    ):
        self.model.zero_grad(
            set_to_none=True
        )

        logits = self.model(
            image
        )

        if class_index is None:
            class_index = (
                logits.argmax(
                    dim=1
                ).item()
            )

        score = logits[
            0,
            class_index
        ]

        score.backward()

        gradients = (
            self.activations
            .grad
        )

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        cam = (
            weights
            * self.activations
        ).sum(
            dim=1,
            keepdim=True
        )

        cam = F.relu(
            cam
        )

        cam = F.interpolate(
            cam,
            size=image.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        cam = (
            cam
            - cam.amin(
                dim=(2, 3),
                keepdim=True
            )
        )

        cam = (
            cam
            / (
                cam.amax(
                    dim=(2, 3),
                    keepdim=True
                )
                + 1e-8
            )
        )

        return (
            cam.detach(),
            logits.detach(),
            class_index
        )

    def remove(self):
        self.handle.remove()


# 28. Generating Grad-CAM

We will target:

```python
model.conv3
```

the last convolutional layer.


In [ ]:
gradcam = GradCAM(
    model,
    model.conv3
)

cam, logits, class_index = (
    gradcam.generate(
        example_image
    )
)

print(
    "CAM:",
    cam.shape
)

print(
    "Class:",
    class_index
)


# 29. Visualizing Grad-CAM Beside the Input


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(8, 4)
)

axes[0].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Input"
)

axes[0].axis(
    "off"
)

axes[1].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[1].imshow(
    cam[
        0,
        0
    ].numpy(),
    cmap="jet",
    alpha=0.45
)

axes[1].set_title(
    f"Grad-CAM — Class {class_index}"
)

axes[1].axis(
    "off"
)

plt.tight_layout()
plt.show()


# 30. Always Remove Debug Hooks When Finished

Hooks remain attached until removed.

Use:

```python
gradcam.remove()
```

when the explanation object is no longer needed.


In [ ]:
gradcam.remove()

print(
    "Grad-CAM hook removed."
)


# 31. Grad-CAM Is Class-Specific

A useful property of Grad-CAM is that we can ask:

> What spatial evidence supports class 0?

or:

> What spatial evidence supports class 2?

for the same image.

Different target classes can produce different maps.


In [ ]:
gradcam = GradCAM(
    model,
    model.conv3
)

cams = []

for class_index in range(3):
    cam, _, _ = gradcam.generate(
        example_image,
        class_index=class_index
    )

    cams.append(
        cam
    )

gradcam.remove()

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

for class_index, axis in enumerate(
    axes
):
    axis.imshow(
        example_image[
            0,
            0
        ].numpy(),
        cmap="gray"
    )

    axis.imshow(
        cams[
            class_index
        ][
            0,
            0
        ].numpy(),
        cmap="jet",
        alpha=0.45
    )

    axis.set_title(
        f"Target class {class_index}"
    )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 32. Target-Layer Choice Changes Grad-CAM

If we generate Grad-CAM from:

- `conv1`
- `conv2`
- `conv3`

the maps will differ.

Earlier layers have finer spatial resolution.

Later layers have more task-specific representation.

There is no single layer that is universally "the explanation."


# 33. Comparing Grad-CAM Layers


In [ ]:
target_layers = {
    "conv1":
        model.conv1,

    "conv2":
        model.conv2,

    "conv3":
        model.conv3
}

layer_cams = {}

for name, layer in (
    target_layers.items()
):
    explainer = GradCAM(
        model,
        layer
    )

    cam, _, _ = (
        explainer.generate(
            example_image
        )
    )

    layer_cams[
        name
    ] = cam

    explainer.remove()

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

for axis, (
    name,
    cam
) in zip(
    axes,
    layer_cams.items()
):
    axis.imshow(
        example_image[
            0,
            0
        ].numpy(),
        cmap="gray"
    )

    axis.imshow(
        cam[
            0,
            0
        ].numpy(),
        cmap="jet",
        alpha=0.45
    )

    axis.set_title(
        name
    )

    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 34. Heatmap Resolution Can Be Misleading

A late CNN layer may have spatial size:

$$
16\times16
$$

or even smaller.

Grad-CAM resizes this coarse map to:

$$
64\times64
$$

or:

$$
224\times224
$$

The smooth-looking heatmap can therefore imply more spatial precision than the network actually had.

Never interpret a coarse Grad-CAM map as exact pixel-level localization.


# 35. Occlusion Sensitivity

Occlusion sensitivity uses a different idea:

> Hide part of the image and measure how much the prediction score changes.

If covering a region causes the target score to drop sharply, that region may be important to the model.


# 36. Occlusion Algorithm

For each spatial patch:

1. Copy the image
2. Replace one patch with a baseline value
3. Run the model
4. Measure score change

Conceptually:

$$
\boxed{
Importance
=
Original\ Score
-
Occluded\ Score
}
$$


# 37. Implementing Occlusion Sensitivity


In [ ]:
def occlusion_sensitivity(
    model,
    image,
    class_index=None,
    patch_size=12,
    stride=6,
    baseline=0.0
):
    model.eval()

    with torch.inference_mode():
        original_logits = model(
            image
        )

        if class_index is None:
            class_index = (
                original_logits.argmax(
                    dim=1
                ).item()
            )

        original_score = (
            original_logits[
                0,
                class_index
            ].item()
        )

    height = image.shape[-2]
    width = image.shape[-1]

    heatmap = torch.zeros(
        height,
        width
    )

    counts = torch.zeros(
        height,
        width
    )

    for y in range(
        0,
        height,
        stride
    ):
        for x in range(
            0,
            width,
            stride
        ):
            y1 = min(
                height,
                y + patch_size
            )

            x1 = min(
                width,
                x + patch_size
            )

            occluded = (
                image
                .detach()
                .clone()
            )

            occluded[
                :,
                :,
                y:y1,
                x:x1
            ] = baseline

            with torch.inference_mode():
                occluded_score = model(
                    occluded
                )[
                    0,
                    class_index
                ].item()

            drop = (
                original_score
                - occluded_score
            )

            heatmap[
                y:y1,
                x:x1
            ] += drop

            counts[
                y:y1,
                x:x1
            ] += 1

    heatmap = (
        heatmap
        / counts.clamp_min(
            1
        )
    )

    return (
        heatmap,
        class_index,
        original_score
    )


# 38. Running Occlusion Sensitivity


In [ ]:
occlusion_map, occ_class, original_score = (
    occlusion_sensitivity(
        model,
        example_image,
        patch_size=12,
        stride=6
    )
)

print(
    "Class:",
    occ_class
)

print(
    "Original score:",
    original_score
)


# 39. Visualizing Occlusion Sensitivity


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(8, 4)
)

axes[0].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Input"
)

axes[0].axis(
    "off"
)

axes[1].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

overlay = axes[1].imshow(
    occlusion_map.numpy(),
    cmap="hot",
    alpha=0.5
)

axes[1].set_title(
    "Occlusion sensitivity"
)

axes[1].axis(
    "off"
)

plt.colorbar(
    overlay,
    ax=axes[1],
    fraction=0.046
)

plt.tight_layout()
plt.show()


# 40. Occlusion Sensitivity Has Its Own Assumptions

Results depend on:

- Patch size
- Stride
- Baseline value
- Whether the occlusion looks realistic

Replacing a patch with zero may create an unnatural image.

The model's reaction can reflect:

- Missing evidence
- Or sensitivity to the artificial occlusion itself

So occlusion maps must also be interpreted carefully.


# 41. Comparing Explanation Methods

Different methods answer different questions.

$$
\begin{array}{|c|c|}
\hline
\textbf{Method} & \textbf{Main Idea} \\
\hline
Feature\ maps &
\text{What activates inside a layer?} \\
\hline
Input\ saliency &
\text{Which pixels have large local score gradients?} \\
\hline
Grad-CAM &
\text{Which deep spatial features support a class score?} \\
\hline
Occlusion &
\text{How does masking a region change the score?} \\
\hline
\end{array}
$$

Agreement between methods can be informative, but disagreement is also useful diagnostic evidence.


# 42. Confidence vs Explanation

Suppose the model predicts:

$$
P(class=2)=0.99
$$

That tells us the model is highly confident.

It does **not** tell us:

- The prediction is correct
- The reasoning is clinically valid
- The heatmap is trustworthy
- The model used the intended anatomy


# 43. Confidence Can Be Wrong

Deep networks can make:

> High-confidence errors

These are especially important to inspect.

A high-confidence error with a suspicious heatmap may reveal shortcut learning.


In [ ]:
model.eval()

with torch.inference_mode():
    logits = model(
        example_image
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

print(
    "Probabilities:",
    probabilities
)

print(
    "Maximum confidence:",
    probabilities.max().item()
)


# 44. Explanation Is Not Confidence

Confidence answers:

> How strongly does the model prefer one prediction?

Explanation methods attempt to answer:

> What image information is associated with that preference?

These are different quantities.


# 45. Failure-Case Inspection

After evaluating a model, collect examples such as:

- False positives
- False negatives
- High-confidence errors
- Low-confidence correct predictions
- Site-specific failures

Then inspect:

- Original image
- Prediction
- Confidence
- Grad-CAM
- Saliency
- Occlusion map


# 46. A Failure-Case Record

A useful analysis table might contain:

$$
\begin{array}{|c|c|}
\hline
Sample\ ID & \text{Stable identifier} \\
\hline
True\ Label & \text{Ground truth} \\
\hline
Prediction & \text{Predicted class} \\
\hline
Confidence & \text{Predicted probability} \\
\hline
Site & \text{Acquisition site} \\
\hline
Device & \text{Scanner/device} \\
\hline
Explanation & \text{Saved heatmap/path} \\
\hline
\end{array}
$$


# 47. Shortcut Learning

Shortcut learning occurs when a model uses an easy predictive signal instead of the intended underlying feature.

Examples:

- Hospital-specific border
- Text label
- Image orientation
- Scanner marker
- Acquisition protocol
- Background artifact

The shortcut may correlate strongly with the label in training data but fail elsewhere.


# 48. Creating a Synthetic Shortcut

We will add a tiny bright square in the top-left corner for one class.

This marker is much easier to detect than the underlying shape.

A model may learn to rely on it.


In [ ]:
def add_corner_shortcut(
    image,
    class_index
):
    image = image.clone()

    if class_index == 2:
        image[
            :,
            2:8,
            2:8
        ] = 1.0

    return image


# 49. Visualizing the Shortcut


In [ ]:
base_image = make_pattern_image(
    2
)

shortcut_image = (
    add_corner_shortcut(
        base_image,
        class_index=2
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7, 3)
)

axes[0].imshow(
    base_image.squeeze(0).numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Original class signal"
)

axes[0].axis(
    "off"
)

axes[1].imshow(
    shortcut_image.squeeze(0).numpy(),
    cmap="gray"
)

axes[1].set_title(
    "With corner shortcut"
)

axes[1].axis(
    "off"
)

plt.tight_layout()
plt.show()


# 50. Spurious Correlation

A **spurious correlation** is a relationship that helps prediction in the current dataset but is not the intended causal/task-relevant signal.

For example:

$$
\boxed{
Corner\ Marker
\leftrightarrow
Class\ 2
}
$$

may exist in training data.

The model can exploit it even though the marker is unrelated to the true object pattern.


# 51. Why Shortcut Learning Is Dangerous

If the shortcut disappears at deployment:

$$
training\ distribution
\neq
deployment\ distribution
$$

performance can collapse.

This is one reason external validation is important.


# 52. Interpretation Can Reveal Shortcuts

If Grad-CAM repeatedly highlights:

- Text
- Borders
- Corners
- Device labels

instead of the intended anatomy, that is a warning.

It does not automatically prove the model is entirely driven by the shortcut.

But it is strong evidence that the pipeline deserves investigation.


# 53. Counterfactual Debugging

One useful debugging idea is:

> Remove or alter the suspicious region and observe whether the prediction changes.

For example:

1. Original image → high class-2 score
2. Remove corner marker
3. Re-run model
4. Compare score

This tests whether the suspicious feature affects the model prediction.


# 54. Explanation Stability

A useful explanation should ideally not change dramatically under tiny irrelevant perturbations.

Possible stability checks:

- Add very small noise
- Slightly resize/crop
- Repeat explanation
- Compare neighboring frames

Large explanation changes under tiny perturbations can indicate instability.


# 55. Saliency Stability Example


In [ ]:
def input_saliency(
    model,
    image,
    class_index=None
):
    x = (
        image
        .detach()
        .clone()
    )

    x.requires_grad_(
        True
    )

    model.zero_grad(
        set_to_none=True
    )

    logits = model(
        x
    )

    if class_index is None:
        class_index = (
            logits.argmax(
                dim=1
            ).item()
        )

    logits[
        0,
        class_index
    ].backward()

    saliency = (
        x.grad
        .detach()
        .abs()
    )

    saliency = (
        saliency
        / (
            saliency.max()
            + 1e-8
        )
    )

    return (
        saliency,
        class_index
    )

saliency_a, saliency_class = (
    input_saliency(
        model,
        example_image
    )
)

perturbed_image = (
    example_image
    + 0.01
    * torch.randn_like(
        example_image
    )
).clamp(
    0.0,
    1.0
)

saliency_b, _ = input_saliency(
    model,
    perturbed_image,
    class_index=saliency_class
)

difference = (
    saliency_a
    - saliency_b
).abs().mean().item()

print(
    "Mean saliency difference:",
    difference
)


# 56. Explanation Method Choice Matters

Different methods can disagree because they use different definitions of importance.

Examples:

- Gradient sensitivity
- Deep feature weighting
- Masking/perturbation

There is no single universal explanation map that reveals "the truth."


# 57. Post-Hoc Explanation

Methods such as:

- Saliency
- Grad-CAM
- Occlusion

are usually **post-hoc**.

The model is trained first.

Then we analyze it afterward.

Post-hoc explanations do not constrain the model to reason in a particular way during training.


# 58. Explainability Does Not Prove Causality

Suppose a heatmap highlights a lesion.

We still cannot conclude:

> The lesion caused the prediction in a clinically meaningful causal sense.

The heatmap is derived from the model's internal behavior.

It is not a causal experiment on the disease process.


# 59. Heatmaps Can Be Overinterpreted

A colorful overlay is visually persuasive.

That can make it easy to overinterpret.

Questions to ask:

1. Which method generated it?
2. Which class was targeted?
3. Which layer was used?
4. How was the map normalized?
5. How coarse was the original map?
6. Is the explanation stable?
7. Does masking the region change the prediction?
8. Does the method behave similarly on errors?


# 60. Normalization Can Change Visual Appearance

Explanation maps are often normalized to:

$$
[0,1]
$$

independently for every image.

That means:

> A bright red region does not necessarily correspond to the same absolute importance across two different images.

Per-image normalization improves visualization but can reduce quantitative comparability.


# 61. Colormap Choice Matters

Some colormaps can exaggerate visual differences.

When possible:

- Show the original image
- Show the raw/normalized explanation separately
- State the colormap
- Avoid implying precise boundaries

Visualization design can affect interpretation.


# 62. Upsampling Can Create False Precision

If a Grad-CAM map is originally:

$$
16\times16
$$

and we resize it to:

$$
256\times256
$$

the smooth boundaries are interpolated.

The underlying explanation did not suddenly gain pixel-level resolution.


# 63. Explanations Can Be Correct-Looking Even for Wrong Predictions

A wrong prediction may still highlight the anatomically plausible region.

This can happen because:

- The model extracted relevant structure but classified it incorrectly
- The heatmap method is too coarse
- Multiple classes depend on the same region

Therefore:

> Plausible localization is not proof of correct classification.


# 64. Explanations Can Look Wrong Even for Correct Predictions

A correct prediction with a strange explanation may reveal:

- Shortcut learning
- Context dependence
- Poor explanation method
- Heatmap instability

Correct classification alone does not guarantee a valid decision process.


# 65. Ultrasound-Specific Interpretation Concerns

Ultrasound interpretation is especially challenging because images contain:

- Speckle
- Acoustic shadows
- Gain-dependent intensity patterns
- Device-specific rendering
- Probe markers
- Depth markers
- Measurement calipers
- Text overlays
- Multiple views
- Operator-dependent acquisition


# 66. Ultrasound Overlays

A model may use:

- Measurement text
- Caliper locations
- Machine annotations
- Hospital labels

if those correlate with diagnosis.

Interpretation maps can help identify such behavior.

But the stronger prevention strategy is:

> Design the dataset and preprocessing so these shortcuts are minimized from the beginning.


# 67. Probe and Orientation Markers

Orientation markers may encode:

- View type
- Probe position
- Acquisition protocol

Sometimes this information is legitimately relevant.

Sometimes it becomes a shortcut.

Whether it should remain in the image depends on the actual intended deployment task.


# 68. Acoustic Artifacts Can Be Clinically Meaningful

Not every artifact should be removed.

Some ultrasound phenomena such as:

- Shadowing
- Enhancement
- Reverberation

may carry diagnostic information.

The goal is not to remove all non-anatomical visual structure.

The goal is to distinguish:

> Task-relevant acoustic information

from:

> Accidental dataset-specific shortcuts.


# 69. Multiple Frames Per Patient

If many frames come from the same ultrasound examination, explanation maps across neighboring frames can help study consistency.

But remember:

- Frames are correlated
- Per-frame explanations are not independent evidence
- Patient-level split design remains essential


# 70. Ultrasound Explanation Stability Across Frames

For a cine loop or repeated frames, ask:

- Does the highlighted region move consistently with anatomy?
- Does the explanation jump to unrelated borders?
- Does the model rely on static overlays?
- Are predictions stable across adjacent frames?

Temporal consistency can be a useful debugging signal.


# 71. Comparing Sites and Devices

For each site/device, inspect:

- Performance
- Prediction confidence
- Explanation patterns
- Common failure modes

If one device produces systematically different explanation maps, investigate preprocessing and domain shift.


# 72. Explainability for Data Harmonization Debugging

If images come from multiple scanners or acquisition settings, explanation maps can help ask:

> Is the classifier attending to anatomy or to scanner-specific appearance?

This does **not** prove successful harmonization.

But it can reveal suspicious scanner-dependent behavior that deserves quantitative follow-up.


# 73. Explanation Should Be Paired With Quantitative Tests

If a model appears to focus on a suspicious corner:

Do not stop at the heatmap.

Also test:

1. Mask the corner
2. Retrain after removing it
3. Evaluate external/site-held-out performance
4. Measure prediction change
5. Compare with control regions

Interpretation should generate testable hypotheses.


# 74. Explanation as a Debugging Loop

A productive workflow is:

$$
\boxed{
Evaluate
\rightarrow
Find\ Failure
\rightarrow
Explain
\rightarrow
Form\ Hypothesis
\rightarrow
Test\ Hypothesis
\rightarrow
Improve\ Pipeline
}
$$


# 75. Sanity Check — Does the Explanation Depend on the Model?

An explanation method should ideally reflect learned model behavior.

A useful sanity check is to compare explanations from:

- Trained model
- Randomly initialized model

If the maps are nearly identical, the method may be showing mostly image structure rather than learned decision logic.


# 76. Random-Model Saliency Comparison


In [ ]:
random_model = ExplainableCNN(
    num_classes=3
)

trained_saliency, trained_class = (
    input_saliency(
        model,
        example_image
    )
)

random_saliency, _ = input_saliency(
    random_model,
    example_image,
    class_index=trained_class
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8, 4)
)

axes[0].imshow(
    trained_saliency[
        0,
        0
    ].numpy(),
    cmap="hot"
)

axes[0].set_title(
    "Trained model"
)

axes[0].axis(
    "off"
)

axes[1].imshow(
    random_saliency[
        0,
        0
    ].numpy(),
    cmap="hot"
)

axes[1].set_title(
    "Random model"
)

axes[1].axis(
    "off"
)

plt.tight_layout()
plt.show()


# 77. Sanity Check — Does the Explanation Change With Target Class?

A class-specific method such as Grad-CAM should often change when the target class changes.

If every class produces nearly identical maps, investigate:

- Model architecture
- Target layer
- Implementation
- Whether the model uses common features for all classes


# 78. Sanity Check — Occlusion of the Highlighted Region

If a Grad-CAM region appears important, mask it and measure the target score.

If the score barely changes, the heatmap may not represent strong causal influence on the output.

This is still not a perfect causal test, but it is a useful consistency check.


# 79. Explanation Reproducibility

For deterministic models in evaluation mode:

- Grad-CAM should usually be reproducible for the same input and target
- Saliency should usually be reproducible for the same computation

But stochastic preprocessing or training-mode dropout can change explanations.

Always use:

```python
model.eval()
```

for standard explanation analysis unless you intentionally study stochastic behavior.


# 80. Common Mistake — Using `torch.inference_mode()` for Gradient Explanations

Methods such as:

- Saliency
- Grad-CAM

need gradients.

Do **not** wrap them in:

```python
torch.inference_mode()
```

or:

```python
torch.no_grad()
```

during the gradient calculation.


# 81. Common Mistake — Forgetting `requires_grad=True` on the Input

For input-gradient saliency:

```python
image.requires_grad_(True)
```

must be enabled on the input tensor whose gradient you want.


# 82. Common Mistake — Forgetting to Clear Gradients

Before repeated explanation calls, clear model gradients:

```python
model.zero_grad(
    set_to_none=True
)
```

Otherwise gradients can accumulate and confuse debugging.


# 83. Common Mistake — Targeting the Wrong Class

Grad-CAM and saliency are class-specific.

Always record:

- Predicted class
- Explained class

They may intentionally differ.

For example, you may want to explain why the model considered an alternative class.


# 84. Common Mistake — Choosing a Non-Spatial Layer for Grad-CAM

Grad-CAM needs a layer with spatial feature maps.

A final linear layer has shape:

$$
(N,\ C)
$$

and no spatial dimensions.

Choose a convolutional feature layer such as:

$$
(N,\ Channels,\ H,\ W)
$$


# 85. Common Mistake — Treating Heatmap Color as a Quantitative Probability

A red pixel does **not** mean:

> 90% probability this location caused the prediction.

The heatmap scale is method-specific and usually normalized for visualization.


# 86. Common Mistake — Comparing Independently Normalized Heatmaps Quantitatively

If every map is normalized separately to:

$$
[0,1]
$$

then peak intensity becomes 1 for every image.

You cannot directly conclude that one image has stronger evidence merely because both contain equally red regions.


# 87. Common Mistake — Cherry-Picking Good-Looking Explanations

Do not show only examples where the heatmap looks clinically plausible.

Inspect a representative set including:

- Correct predictions
- Errors
- Multiple classes
- Multiple sites
- Multiple devices
- Easy and difficult cases


# 88. Common Mistake — Using Explainability as Proof of Safety

An appealing heatmap does not prove:

- Robustness
- Fairness
- Calibration
- Clinical validity
- Generalization
- Safety

These require dedicated evaluation.


# 89. Common Mistake — Ignoring Preprocessing

If the model receives:

- Cropped images
- Resized images
- Normalized images

then explanation coordinates correspond to the model input representation.

When overlaying explanations on the original image, spatial transforms must be reversed correctly.


# 90. Common Mistake — Interpreting Resized Heatmaps as Segmentation

Grad-CAM is not a segmentation mask.

It usually provides coarse localization of class-relevant regions.

Do not calculate lesion boundaries from Grad-CAM unless the method and task are explicitly validated for that purpose.


# 91. A Reusable Explanation Report

For one sample, it is useful to collect:

- Image
- True label
- Predicted class
- Confidence
- Saliency
- Grad-CAM
- Occlusion map

This gives multiple views of the same decision.


In [ ]:
def explanation_report_data(
    model,
    image,
    true_label=None
):
    model.eval()

    with torch.inference_mode():
        logits = model(
            image
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predicted_class = (
            logits.argmax(
                dim=1
            ).item()
        )

        confidence = (
            probabilities.max().item()
        )

    saliency, _ = input_saliency(
        model,
        image,
        class_index=predicted_class
    )

    gradcam = GradCAM(
        model,
        model.conv3
    )

    cam, _, _ = gradcam.generate(
        image,
        class_index=predicted_class
    )

    gradcam.remove()

    occlusion_map, _, _ = (
        occlusion_sensitivity(
            model,
            image,
            class_index=predicted_class,
            patch_size=12,
            stride=8
        )
    )

    return {
        "true_label":
            true_label,

        "predicted_class":
            predicted_class,

        "confidence":
            confidence,

        "saliency":
            saliency,

        "gradcam":
            cam,

        "occlusion":
            occlusion_map
    }

report = explanation_report_data(
    model,
    example_image,
    true_label=2
)

print(
    "Prediction:",
    report["predicted_class"]
)

print(
    "Confidence:",
    report["confidence"]
)


# 92. Plotting a Multi-Method Explanation Report


In [ ]:
fig, axes = plt.subplots(
    1,
    4,
    figsize=(14, 4)
)

axes[0].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Input"
)

axes[1].imshow(
    report["saliency"][
        0,
        0
    ].numpy(),
    cmap="hot"
)

axes[1].set_title(
    "Saliency"
)

axes[2].imshow(
    example_image[
        0,
        0
    ].numpy(),
    cmap="gray"
)

axes[2].imshow(
    report["gradcam"][
        0,
        0
    ].numpy(),
    cmap="jet",
    alpha=0.45
)

axes[2].set_title(
    "Grad-CAM"
)

axes[3].imshow(
    report["occlusion"].numpy(),
    cmap="hot"
)

axes[3].set_title(
    "Occlusion"
)

for axis in axes:
    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 93. What If Explanation Methods Disagree?

That is not automatically a problem.

Different methods measure different notions of influence.

Ask:

- Is the disagreement systematic?
- Does one method align with perturbation tests?
- Is one method unstable?
- Does the target layer matter?
- Are explanations sensitive to preprocessing?

Disagreement can reveal uncertainty about the model's decision mechanism.


# 94. Quantitative Explanation Evaluation Is Hard

For some tasks, explanations can be compared with:

- Segmentation masks
- Bounding boxes
- Expert annotations

But overlap with an annotation does not automatically prove the explanation is correct.

The model may use:

- Context outside the lesion
- Global shape
- Acquisition cues

Explanation evaluation requires a clearly defined objective.


# 95. Explanation and Segmentation Are Different Tasks

A classifier answers:

> Which class?

A segmenter answers:

> Which pixels belong to a region?

Grad-CAM from a classifier is not trained to produce exact segmentation.

So coarse localization should not be treated as pixel-accurate anatomy.


# 96. Interpretability for Model Comparison

Suppose two models have similar AUROC.

Interpretation may help compare whether they use:

- Similar regions
- Different shortcuts
- Different site artifacts

This can guide further experiments.

But model selection should still rely on predefined quantitative criteria.


# 97. Using Explanations During Dataset Debugging

Interpretability can reveal:

- Cropping mistakes
- Black borders
- Padding artifacts
- Text leakage
- Incorrect masks
- Wrong view labels
- Unexpected acquisition cues

This is one of the highest-value uses of explainability in practice.


# 98. Using Explanations During Preprocessing Debugging

Suppose a crop removes the true anatomy but preserves a label marker.

The model may still perform suspiciously well.

A heatmap focused on the marker can expose the preprocessing failure.

Again:

> The explanation should trigger a concrete pipeline check.


# 99. Using Explanations During External Validation

If performance drops on a new hospital:

Compare explanation maps between:

- Development site
- External site

Questions:

- Is the model focusing on different regions?
- Are new borders or text present?
- Did preprocessing change?
- Did image scale change?


# 100. Explainability Workflow for Ultrasound

A strong analysis workflow may be:

1. Train with patient-level splits
2. Evaluate standard metrics
3. Identify errors
4. Generate explanations
5. Inspect overlays and artifacts
6. Compare sites/devices
7. Form hypotheses
8. Test them with masking/retraining/external validation


# 101. The Most Important Explainability Principle

Use explanations to ask:

> **What should I test next?**

not:

> **What has been proven?**

This mindset prevents overconfidence.


# 102. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Pass an image through the first convolution of a CNN and visualize 4 feature maps.

## Exercise 2

Compute the gradient of a selected class logit with respect to the input image.

## Exercise 3

Convert the input gradient into an absolute saliency map.

## Exercise 4

Implement Grad-CAM for a chosen convolutional layer.

## Exercise 5

Generate Grad-CAM for two different target classes.

## Exercise 6

Compare Grad-CAM from an early and late convolutional layer.

## Exercise 7

Implement occlusion sensitivity using a 10×10 patch.

## Exercise 8

Add an artificial corner shortcut to one class and explain why this can be dangerous.

## Exercise 9

Compare explanations for an original image and a slightly perturbed image.

## Exercise 10

Create a multi-panel explanation report containing:
- Input
- Saliency
- Grad-CAM
- Occlusion map


# 103. Conceptual Challenges

## Challenge 1

Why can a highly accurate model still need interpretation?

## Challenge 2

What does an input-gradient saliency map measure?

## Challenge 3

Why can raw saliency maps be noisy?

## Challenge 4

What information does Grad-CAM use?

## Challenge 5

Why is Grad-CAM usually applied to a convolutional layer rather than a final linear layer?

## Challenge 6

Why can a resized Grad-CAM map appear more precise than it really is?

## Challenge 7

What does occlusion sensitivity measure?

## Challenge 8

Why can occlusion itself create an out-of-distribution artifact?

## Challenge 9

What is shortcut learning?

## Challenge 10

What is a spurious correlation?

## Challenge 11

Why is confidence not the same as explanation quality?

## Challenge 12

Why do explainability methods not prove causality?

## Challenge 13

Why should explanations be inspected on incorrect predictions too?

## Challenge 14

Why are ultrasound overlays a potential leakage source?

## Challenge 15

Why should explanation findings lead to additional quantitative tests?


# 104. Exercise Solutions


In [ ]:
# Exercise 1
exercise_image = make_pattern_image(
    0
).unsqueeze(
    0
)

model.eval()

with torch.inference_mode():
    exercise_features = F.relu(
        model.conv1(
            exercise_image
        )
    )

print(
    "Exercise 1 feature shape:",
    exercise_features.shape
)


In [ ]:
# Exercise 2 and 3
exercise_input = (
    exercise_image
    .detach()
    .clone()
)

exercise_input.requires_grad_(
    True
)

model.zero_grad(
    set_to_none=True
)

exercise_logits = model(
    exercise_input
)

exercise_class = (
    exercise_logits.argmax(
        dim=1
    ).item()
)

exercise_logits[
    0,
    exercise_class
].backward()

exercise_saliency = (
    exercise_input.grad
    .detach()
    .abs()
)

exercise_saliency = (
    exercise_saliency
    / (
        exercise_saliency.max()
        + 1e-8
    )
)

print(
    "Exercise 3:",
    exercise_saliency.shape
)


In [ ]:
# Exercise 4 and 5
exercise_gradcam = GradCAM(
    model,
    model.conv3
)

exercise_cam_0, _, _ = (
    exercise_gradcam.generate(
        exercise_image,
        class_index=0
    )
)

exercise_cam_1, _, _ = (
    exercise_gradcam.generate(
        exercise_image,
        class_index=1
    )
)

exercise_gradcam.remove()

print(
    "Exercise 4/5:",
    exercise_cam_0.shape,
    exercise_cam_1.shape
)


In [ ]:
# Exercise 6
early_explainer = GradCAM(
    model,
    model.conv1
)

late_explainer = GradCAM(
    model,
    model.conv3
)

early_cam, _, _ = (
    early_explainer.generate(
        exercise_image
    )
)

late_cam, _, _ = (
    late_explainer.generate(
        exercise_image
    )
)

early_explainer.remove()
late_explainer.remove()

print(
    "Exercise 6:",
    early_cam.shape,
    late_cam.shape
)


In [ ]:
# Exercise 7
exercise_occlusion, _, _ = (
    occlusion_sensitivity(
        model,
        exercise_image,
        patch_size=10,
        stride=5
    )
)

print(
    "Exercise 7:",
    exercise_occlusion.shape
)

# Exercise 8
shortcut_example = add_corner_shortcut(
    exercise_image[
        0
    ],
    class_index=2
)

print(
    "Exercise 8:",
    shortcut_example.shape
)


# 105. Key Takeaways

In this notebook, we learned:

- Why model interpretation matters
- Why interpretation and evaluation are different
- CNN feature maps
- Activation visualization
- Input gradients
- Saliency maps
- Grad-CAM intuition
- Grad-CAM mathematics
- Implementing Grad-CAM
- Target-layer choice
- Heatmap-resolution limitations
- Occlusion sensitivity
- Confidence vs explanation
- Failure-case inspection
- Shortcut learning
- Spurious correlations
- Counterfactual debugging
- Explanation stability
- Post-hoc explanations
- Explainability vs causality
- Heatmap normalization pitfalls
- Ultrasound-specific concerns
- Overlay leakage
- Device/site differences
- Acoustic artifacts
- Explanation sanity checks
- Multi-method explanation reports
- Using explanations as debugging tools

The core Grad-CAM idea is:

$$
\boxed{
Class\ Gradient
\rightarrow
Feature\ Map\ Weights
\rightarrow
Weighted\ Spatial\ Activation
\rightarrow
Heatmap
}
$$

The central interpretation principle is:

$$
\boxed{
\text{Explanation}
\neq
\text{Proof of Causality}
}
$$

A better mindset is:

$$
\boxed{
\text{Explanation}
\rightarrow
\text{Hypothesis}
\rightarrow
\text{Quantitative Test}
}
$$


# 106. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why can a model with high accuracy still use the wrong visual cues?
2. What is a feature map?
3. What does an activation map show?
4. What does input-gradient saliency measure?
5. Why must the input require gradients for saliency?
6. Why should model gradients be cleared before explanation?
7. What does Grad-CAM stand for?
8. What are the Grad-CAM channel weights?
9. Why is ReLU applied to the Grad-CAM combination?
10. Why is the target convolutional layer important?
11. Why is Grad-CAM spatially coarse?
12. What does occlusion sensitivity measure?
13. Why can the occlusion baseline matter?
14. Why does high confidence not imply a valid explanation?
15. What is shortcut learning?
16. What is a spurious correlation?
17. Why can text overlays be dangerous in medical images?
18. Why can explanation maps differ between target classes?
19. Why can explanation methods disagree?
20. Why is per-image normalization of heatmaps a comparison problem?
21. Why should explanations be inspected on false positives and false negatives?
22. Why can a correct prediction still have a suspicious explanation?
23. Why can a wrong prediction still highlight a plausible anatomical region?
24. Why should explanations be compared across sites/devices?
25. Why do explanations not prove causal reasoning?
26. What is a useful explanation sanity check?
27. Why should a suspicious heatmap lead to masking or retraining experiments?
28. Why is Grad-CAM not a segmentation method?
29. How can explanations help debug harmonization or preprocessing?
30. What is the safest general role for explainability in a deep-learning workflow?


# Next Notebook

# 23 — End-to-End PyTorch Project: From Data to Reliable Model

In the next notebook, we will study:

- Defining the prediction task
- Designing train / validation / test splits
- Building a reusable dataset pipeline
- Creating a baseline model
- Training and validation
- Choosing metrics
- Checkpointing
- Hyperparameter experiments
- Transfer-learning baseline
- Error analysis
- Explainability checks
- Reproducibility
- Saving predictions and experiment metadata
- Final test evaluation
- Organizing a complete PyTorch project
- Preparing the workflow for a real ultrasound classification project
